In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 29. 텐서 병렬과 파이프라인 병렬

> **학습 목표**
> - 텐서 병렬(TP)이 행렬을 어떻게 분할하는지 이해한다
> - 파이프라인 병렬(PP)의 버블(bubble) 문제를 안다
> - 1F1B 스케줄링을 설명한다

## 29.1 왜 모델 병렬이 필요한가

데이터 병렬: 모델이 단일 GPU에 들어가야 함. 70B+ 모델은 안 들어감.

해결:
- **텐서 병렬 (TP)**: 한 층의 가중치를 여러 GPU에 분할
- **파이프라인 병렬 (PP)**: 층을 여러 GPU에 분할
- **3D 병렬**: DP + TP + PP 조합 (GPT-3 학습 방식)

## 29.2 텐서 병렬 (Megatron-LM 방식)

선형층 $Y = XW$를 분할:

### Column Parallel (열 분할)
$W = [W_1; W_2]$ (세로로 분할)
$$Y = XW = X[W_1; W_2] = [XW_1, XW_2]$$
- 각 GPU가 $W_i$ 계산, 결과 $XW_i$ 반환
- 출력을 이어붙이려면 All-Gather

### Row Parallel (행 분할)
$W = [W_1, W_2]$ (가로로 분할)
$$Y = XW = [X_1, X_2][W_1; W_2] = X_1 W_1 + X_2 W_2$$
- 각 GPU가 $X_i W_i$ 계산
- All-Reduce로 합산

Megatron: 어텐션은 column, FFN은 column + row 조합으로 All-Reduce 1회.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Column Parallel Linear 시뮬레이션
class ColumnParallelLinear(nn.Module):
    """W를 세로로 분할 (간소화 시뮬레이션)."""
    def __init__(self, in_features, out_features, n_gpus=2):
        super().__init__()
        assert out_features % n_gpus == 0
        self.n_gpus = n_gpus
        self.out_per_gpu = out_features // n_gpus
        # 각 GPU의 Weight
        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(in_features, self.out_per_gpu) * 0.1)
            for _ in range(n_gpus)
        ])

    def forward(self, x):
        # 각 GPU가 자기 Subspace Computation
        outputs = [x @ w for w in self.weights]
        # All-Gather: Result를 이어붙임
        return torch.cat(outputs, dim=-1)

# Row Parallel Linear
class RowParallelLinear(nn.Module):
    """W를 가로로 분할 (Input도 분할)."""
    def __init__(self, in_features, out_features, n_gpus=2):
        super().__init__()
        assert in_features % n_gpus == 0
        self.n_gpus = n_gpus
        self.in_per_gpu = in_features // n_gpus
        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(self.in_per_gpu, out_features) * 0.1)
            for _ in range(n_gpus)
        ])

    def forward(self, x):
        # Input도 분할
        x_chunks = torch.chunk(x, self.n_gpus, dim=-1)
        # 각 GPU Computation
        partial = [xc @ w for xc, w in zip(x_chunks, self.weights)]
        # All-Reduce: Sum산
        return sum(partial)

# Test
d_in, d_out = 64, 64
n_gpus = 2

# Original
torch.manual_seed(0)
W_full = torch.randn(d_in, d_out) * 0.1

# Column parallel 시뮬레이션
col = ColumnParallelLinear(d_in, d_out, n_gpus)
with torch.no_grad():
    # Weight를 W_full의 분할로 Setting
    for i in range(n_gpus):
        col.weights[i].copy_(W_full[:, i*col.out_per_gpu:(i+1)*col.out_per_gpu])

x = torch.randn(4, d_in)
y_full = x @ W_full
y_col = col(x)
print(f"Column Parallel 오차: {(y_full - y_col).abs().max():.2e}")

# Row parallel
row = RowParallelLinear(d_in, d_out, n_gpus)
with torch.no_grad():
    for i in range(n_gpus):
        row.weights[i].copy_(W_full[i*row.in_per_gpu:(i+1)*row.in_per_gpu, :])
y_row = row(x)
print(f"Row Parallel Error: {(y_full - y_row).abs().max():.2e}")
print("\n=> 분할 후 다시 Sum치Plane Original과 동일!")


## 29.3 어텐션의 헤드 분할

MHA는 자연스럽게 TP에 적합: 각 헤드를 다른 GPU에 할당.

- $h$개 헤드, $n$ GPU → 각 GPU에 $h/n$개 헤드
- 헤드별로 $W_i^Q, W_i^K, W_i^V$ 계산 (독립)
- 헤드 결과 Concat → All-Reduce로 결합

## 29.4 파이프라인 병렬

층을 여러 GPU에 분할:
- GPU 0: 층 1-4
- GPU 1: 층 5-8
- GPU 2: 층 9-12

순전파는 GPU 0 → 1 → 2 순서. 역전파는 반대.

### 버블 문제
한 GPU가 일하는 동안 다른 GPU는 대기 → 유휴 시간(버블) 발생.

버블 비율: $\frac{p-1}{m + p - 1}$
- $p$: 파이프라인 단계 수
- $m$: 마이크로배치 수

해결: **마이크로배칭** — 배치를 $m$개로 쪼개어 파이프라인 채움.


In [ ]:
# 파이프라인 버블 시각화
def pipeline_bubble_ratio(p, m):
    """Bubble Ratio."""
    return (p - 1) / (m + p - 1)

# 다양한 p, m에 대해
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bubble Ratio
ax = axes[0]
for p in [2, 4, 8, 16]:
    ms = np.arange(1, 50)
    ratios = [pipeline_bubble_ratio(p, m) for m in ms]
    ax.plot(ms, ratios, label=f'p={p}', linewidth=2)
ax.set_xlabel('마이크로Batch 수 m')
ax.set_ylabel('Bubble Ratio')
ax.set_title('Pipeline Bubble Ratio')
ax.legend(); ax.grid(True, alpha=0.3)

# Gantt 차트 (간소화)
ax = axes[1]
p, m = 4, 8
# 각 GPU의 작업 스케줄 (Forward Pass F, Backward Pass B, 유휴 .)
schedule = []
for stage in range(p):
    s = []
    # forward: stage 순서대로
    for micro in range(m):
        # 시작 Time = stage + micro
        for t in range(stage + micro + 1):
            pass
    # 간단히 forward 후 backward
    s = ['.'] * (2 * m + 2 * (p - 1))
    # forward Pattern
    for micro in range(m):
        start = stage + micro
        if start < len(s):
            s[start] = f'F{micro}'
    # backward
    for micro in range(m):
        start = m + 2 * (p - 1) - stage + micro
        if 0 <= start < len(s):
            s[start] = f'B{micro}'
    schedule.append(s)

# Visualization (간단)
for i, s in enumerate(schedule):
    for j, x in enumerate(s):
        if x.startswith('F'):
            ax.barh(i, 1, left=j, color='blue', alpha=0.7)
        elif x.startswith('B'):
            ax.barh(i, 1, left=j, color='red', alpha=0.7)
        else:
            ax.barh(i, 1, left=j, color='gray', alpha=0.3)
ax.set_xlabel('Time Step')
ax.set_ylabel('GPU (stage)')
ax.set_title(f'Pipeline 스케줄 (p={p}, m={m})')
ax.set_yticks(range(p))
plt.tight_layout()
plt.savefig('../figures/ch29_pipeline_bubble.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"p=4, m=8 버블 비율: {pipeline_bubble_ratio(4, 8)*100:.1f}%")
print(f"p=4, m=32 Bubble Ratio: {pipeline_bubble_ratio(4, 32)*100:.1f}%")
print("=> 마이크로Batch 수가 많을수록 Bubble Decrease.")


## 29.5 1F1B 스케줄링

단순 스케줄: 모든 forward 후 모든 backward → 큰 메모리.

**1F1B**: forward 1개, backward 1개 교대로. 활성값 메모리 절약.


## 29.6 3D 병렬 (DP + TP + PP)

GPT-3 (175B) 학습 방식:
- DP: 데이터 병렬 (배치 분할)
- TP: 어텐션/FFN 가중치 분할
- PP: 층 분할

예: 1024 GPU = 8 DP × 8 TP × 16 PP

## 29.7 핵심 정리

| 병렬 | 분할 단위 | 통신 | 메모리 |
|---|---|---|---|
| 데이터 (DP) | 데이터 | All-Reduce (그래디언트) | 모델 전체 |
| 텐서 (TP) | 가중치 행렬 | All-Reduce / All-Gather | 층 일부 |
| 파이프라인 (PP) | 층 | Point-to-Point | 층 일부 + 활성값 |
| 3D | 모두 | 혼합 | 최소 |

## 연습문제

1. Column parallel과 row parallel을 직접 구현하고 원본과 결과가 같음을 보이라.
2. 파이프라인 버블 비율을 p=2, 4, 8과 m=4, 16, 64에 대해 계산하라.
3. 어텐션 헤드를 4 GPU에 분할하는 텐서 병렬을 시뮬레이션하라.
4. 3D 병렬에서 1024 GPU를 DP=8, TP=8, PP=16으로 구성하는 이유를 설명하라.
5. 1F1B 스케줄링이 메모리를 절약하는 원리를 설명하라.

> 해설: `solutions/ch29_solutions.ipynb`
